## 1. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Portable paths — notebook runs from the notebooks/ directory
PROJECT_ROOT = Path.cwd().parent
DATA_CLEANED = PROJECT_ROOT / 'data' / 'cleaned'
REPORTS = PROJECT_ROOT / 'reports'
MAPPINGS = REPORTS / 'key_mappings'

# Ensure output directories exist
REPORTS.mkdir(exist_ok=True)
MAPPINGS.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Cleaned data: {DATA_CLEANED}")
print(f"Reports: {REPORTS}")

Project root: c:\Users\ASUS-PC\Downloads\Telegram Desktop\Project Data Analysis\Project Data Analysis\Project Data Analysis
Cleaned data: c:\Users\ASUS-PC\Downloads\Telegram Desktop\Project Data Analysis\Project Data Analysis\Project Data Analysis\data\cleaned
Reports: c:\Users\ASUS-PC\Downloads\Telegram Desktop\Project Data Analysis\Project Data Analysis\Project Data Analysis\reports


## 2. Load All Cleaned Datasets

In [2]:
# Load the six cleaned CSVs
medicine = pd.read_csv(DATA_CLEANED / 'medicine_clean.csv')
generic = pd.read_csv(DATA_CLEANED / 'generic_clean.csv')
manufacturer = pd.read_csv(DATA_CLEANED / 'manufacturer_clean.csv')
dosage_form = pd.read_csv(DATA_CLEANED / 'dosage_form_clean.csv')
drug_class = pd.read_csv(DATA_CLEANED / 'drug_class_clean.csv')
indication = pd.read_csv(DATA_CLEANED / 'indication_clean.csv')

# Quick summary of what was loaded
datasets = {
    'medicine': medicine,
    'generic': generic,
    'manufacturer': manufacturer,
    'dosage_form': dosage_form,
    'drug_class': drug_class,
    'indication': indication,
}

for name, df in datasets.items():
    print(f"{name:15s} → {df.shape[0]:>6,} rows × {df.shape[1]:>2} cols")

medicine        → 21,714 rows × 17 cols
generic         →  1,711 rows × 37 cols
manufacturer    →    240 rows ×  5 cols
dosage_form     →    113 rows ×  4 cols
drug_class      →    453 rows ×  4 cols
indication      →  2,043 rows ×  4 cols


## 3. Dataset Inventory



In [3]:
# Detailed inventory: columns, types, nulls, uniques
for name, df in datasets.items():
    print(f"\n{'='*70}")
    print(f"  {name.upper()}")
    print(f"{'='*70}")
    print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"  Duplicate rows: {df.duplicated().sum()}")
    print(f"  Columns:")
    for col in df.columns:
        na = df[col].isna().sum()
        uniq = df[col].nunique()
        dtype = df[col].dtype
        print(f"    {col:45s}  type={str(dtype):8s}  nulls={na:>5}  unique={uniq:>5}")


  MEDICINE
  Shape: 21714 rows × 17 columns
  Duplicate rows: 0
  Columns:
    brand id                                       type=int64     nulls=    0  unique=21714
    brand name                                     type=str       nulls=    0  unique=13934
    type                                           type=str       nulls=    0  unique=    2
    slug                                           type=str       nulls=    0  unique=21591
    dosage form                                    type=str       nulls=    0  unique=  112
    generic                                        type=str       nulls=    2  unique= 1661
    strength                                       type=str       nulls=  849  unique= 1033
    manufacturer                                   type=str       nulls=    0  unique=  232
    package_container_raw                          type=str       nulls=   42  unique= 7364
    Package Size                                   type=str       nulls= 7779  unique= 3531
    

## 4. Primary Key Analysis

In [4]:
# Define candidate primary keys based on actual column names
candidate_pks = {
    'medicine':     'brand id',
    'generic':      'generic id',
    'manufacturer': 'manufacturer id',
    'dosage_form':  'dosage form id',
    'drug_class':   'drug class id',
    'indication':   'indication id',
}

pk_results = []
for name, pk_col in candidate_pks.items():
    df = datasets[name]
    is_unique = df[pk_col].is_unique
    null_count = df[pk_col].isna().sum()
    dup_count = df[pk_col].duplicated().sum()
    dtype = str(df[pk_col].dtype)
    
    # Classification
    if is_unique and null_count == 0:
        status = " Confirmed Primary Key"
    elif null_count == 0:
        status = " Candidate (has duplicates)"
    else:
        status = " Not Reliable"
    
    pk_results.append({
        'dataset': name,
        'column': pk_col,
        'unique': is_unique,
        'nulls': null_count,
        'duplicates': dup_count,
        'dtype': dtype,
        'status': status
    })
    print(f"{name:15s}.{pk_col:20s}  unique={is_unique}  nulls={null_count}  dupes={dup_count}  dtype={dtype}  → {status}")

pk_df = pd.DataFrame(pk_results)
print("\n All six datasets have Confirmed Primary Keys.")

medicine       .brand id              unique=True  nulls=0  dupes=0  dtype=int64  →  Confirmed Primary Key
generic        .generic id            unique=True  nulls=0  dupes=0  dtype=int64  →  Confirmed Primary Key
manufacturer   .manufacturer id       unique=True  nulls=0  dupes=0  dtype=int64  →  Confirmed Primary Key
dosage_form    .dosage form id        unique=True  nulls=0  dupes=0  dtype=int64  →  Confirmed Primary Key
drug_class     .drug class id         unique=True  nulls=0  dupes=0  dtype=int64  →  Confirmed Primary Key
indication     .indication id         unique=True  nulls=0  dupes=0  dtype=int64  →  Confirmed Primary Key

 All six datasets have Confirmed Primary Keys.


## 5. Foreign Key Analysis — Value Matching

In [5]:
# Define the FK relationships to test
fk_tests = [
    ('medicine', 'generic',     'generic',      'generic name',      'Generic'),
    ('medicine', 'manufacturer','manufacturer',  'manufacturer name', 'Manufacturer'),
    ('medicine', 'dosage form', 'dosage_form',   'dosage form name',  'Dosage Form'),
    ('generic',  'drug class',  'drug_class',    'drug class name',   'Drug Class'),
    ('generic',  'indication',  'indication',    'indication name',   'Indication'),
]

fk_results = []
for child_tbl, child_col, parent_tbl, parent_col, label in fk_tests:
    child_df = datasets[child_tbl]
    parent_df = datasets[parent_tbl]
    
    # Get unique non-null values from child and parent
    child_vals = set(child_df[child_col].dropna().str.strip().unique())
    parent_vals = set(parent_df[parent_col].dropna().str.strip().unique())
    
    matched = child_vals & parent_vals
    unmatched = child_vals - parent_vals
    match_pct = len(matched) / len(child_vals) * 100 if child_vals else 0
    
    fk_results.append({
        'relationship': f"{child_tbl}.{child_col} → {parent_tbl}.{parent_col}",
        'child_unique': len(child_vals),
        'parent_unique': len(parent_vals),
        'matched': len(matched),
        'unmatched': len(unmatched),
        'match_pct': f"{match_pct:.2f}%"
    })
    
    print(f"\n--- {label} ---")
    print(f"  {child_tbl}.{child_col} → {parent_tbl}.{parent_col}")
    print(f"  Child unique values:  {len(child_vals)}")
    print(f"  Parent unique values: {len(parent_vals)}")
    print(f"  Matched:              {len(matched)}")
    print(f"  Unmatched (orphans):  {len(unmatched)}")
    print(f"  Match rate:           {match_pct:.2f}%")
    if unmatched:
        print(f"   Unmatched values: {list(unmatched)[:10]}")

print("\nAll relationships show 100% match — no orphan child records.")


--- Generic ---
  medicine.generic → generic.generic name
  Child unique values:  1661
  Parent unique values: 1711
  Matched:              1661
  Unmatched (orphans):  0
  Match rate:           100.00%

--- Manufacturer ---
  medicine.manufacturer → manufacturer.manufacturer name
  Child unique values:  232
  Parent unique values: 240
  Matched:              232
  Unmatched (orphans):  0
  Match rate:           100.00%

--- Dosage Form ---
  medicine.dosage form → dosage_form.dosage form name
  Child unique values:  112
  Parent unique values: 113
  Matched:              112
  Unmatched (orphans):  0
  Match rate:           100.00%

--- Drug Class ---
  generic.drug class → drug_class.drug class name
  Child unique values:  358
  Parent unique values: 421
  Matched:              358
  Unmatched (orphans):  0
  Match rate:           100.00%

--- Indication ---
  generic.indication → indication.indication name
  Child unique values:  672
  Parent unique values: 2043
  Matched:         

## 6. Cardinality Analysis


In [6]:
# --- Medicine ↔ Generic ---
# How many medicines per generic?
med_per_gen = medicine.groupby('generic')['brand id'].nunique()
print("=== Generic → Medicine ===")
print(f"  Min medicines per generic: {med_per_gen.min()}")
print(f"  Max medicines per generic: {med_per_gen.max()}")
print(f"  Mean: {med_per_gen.mean():.1f}")
print(f"  Generics with 1 medicine:  {(med_per_gen == 1).sum()}")
print(f"  Generics with >1 medicine: {(med_per_gen > 1).sum()}")
print(f"  → Cardinality: 1:N (one generic can have MANY branded medicines)")

# Does each medicine have exactly 1 generic?
gen_per_med = medicine.groupby('brand id')['generic'].nunique()
print(f"  Medicines with >1 generic: {(gen_per_med > 1).sum()}")
print(f"  → Each medicine maps to exactly 1 generic ✅")

=== Generic → Medicine ===
  Min medicines per generic: 1
  Max medicines per generic: 341
  Mean: 13.1
  Generics with 1 medicine:  446
  Generics with >1 medicine: 1215
  → Cardinality: 1:N (one generic can have MANY branded medicines)
  Medicines with >1 generic: 0
  → Each medicine maps to exactly 1 generic ✅


In [7]:
# --- Medicine ↔ Manufacturer ---
med_per_mfr = medicine.groupby('manufacturer')['brand id'].nunique()
print("=== Manufacturer → Medicine ===")
print(f"  Min: {med_per_mfr.min()}, Max: {med_per_mfr.max()}, Mean: {med_per_mfr.mean():.1f}")
print(f"  → Cardinality: 1:N")

# --- Medicine ↔ Dosage Form ---
med_per_dos = medicine.groupby('dosage form')['brand id'].nunique()
print("\n=== Dosage Form → Medicine ===")
print(f"  Min: {med_per_dos.min()}, Max: {med_per_dos.max()}, Mean: {med_per_dos.mean():.1f}")
print(f"  → Cardinality: 1:N")

=== Manufacturer → Medicine ===
  Min: 1, Max: 1182, Mean: 93.6
  → Cardinality: 1:N

=== Dosage Form → Medicine ===
  Min: 1, Max: 9329, Mean: 193.9
  → Cardinality: 1:N


In [8]:
# --- Generic ↔ Drug Class ---
gen_dc = generic[['generic id', 'drug class']].dropna(subset=['drug class'])
gen_per_dc = gen_dc.groupby('drug class')['generic id'].nunique()
print("=== Drug Class → Generic ===")
print(f"  Generics with drug class: {len(gen_dc)} (NULL for {generic['drug class'].isna().sum()})")
print(f"  Max generics per drug class: {gen_per_dc.max()}")
print(f"  Drug classes with >1 generic: {(gen_per_dc > 1).sum()}")
print(f"  Each generic has at most 1 drug class value: {generic.groupby('generic id')['drug class'].nunique().max() <= 1}")
print(f"  → Cardinality: 1:N")
print(f"\n  Top 5 drug classes by generic count:")
for dc, cnt in gen_per_dc.sort_values(ascending=False).head(5).items():
    print(f"    '{dc}': {cnt} generics")

=== Drug Class → Generic ===
  Generics with drug class: 1650 (NULL for 61)
  Max generics per drug class: 143
  Drug classes with >1 generic: 249
  Each generic has at most 1 drug class value: True
  → Cardinality: 1:N

  Top 5 drug classes by generic count:
    'Herbal and Nutraceuticals': 143 generics
    'Cytotoxic Chemotherapy': 41 generics
    'Vaccines, Anti-sera & Immunoglobulin': 36 generics
    'Drugs for Osteoarthritis': 27 generics
    'Combined antihypertensive preparations': 20 generics


In [9]:
# --- Generic ↔ Indication ---
gen_ind = generic[['generic id', 'indication']].dropna(subset=['indication'])
gen_per_ind = gen_ind.groupby('indication')['generic id'].nunique()
print("=== Indication → Generic ===")
print(f"  Generics with indication: {len(gen_ind)} (NULL for {generic['indication'].isna().sum()})")
print(f"  Max generics per indication: {gen_per_ind.max()}")
print(f"  Indications with >1 generic: {(gen_per_ind > 1).sum()}")
print(f"  Each generic has at most 1 indication value: {generic.groupby('generic id')['indication'].nunique().max() <= 1}")
print(f"  → Cardinality: 1:N")
print(f"\n  Top 5 indications by generic count:")
for ind_name, cnt in gen_per_ind.sort_values(ascending=False).head(5).items():
    print(f"    '{ind_name}': {cnt} generics")

=== Indication → Generic ===
  Generics with indication: 1627 (NULL for 84)
  Max generics per indication: 35
  Indications with >1 generic: 289
  Each generic has at most 1 indication value: True
  → Cardinality: 1:N

  Top 5 indications by generic count:
    'Urinary tract infection': 35 generics
    'Type 2 DM': 35 generics
    'Hypertension': 28 generics
    'Rheumatoid arthritis': 28 generics
    'Vitamin deficiency': 21 generics


## 7. Referential Integrity — Unused Parent Entities

While no child records are orphaned, some parent entities have **no children** referencing them. These are valid catalog entries — they should be preserved in SQL.

In [10]:
# Find parent entities not referenced by any child
unused = {}

unused['manufacturers'] = set(manufacturer['manufacturer name'].str.strip()) - set(medicine['manufacturer'].dropna().str.strip())
unused['dosage_forms'] = set(dosage_form['dosage form name'].str.strip()) - set(medicine['dosage form'].dropna().str.strip())
unused['generics'] = set(generic['generic name'].str.strip()) - set(medicine['generic'].dropna().str.strip())
unused['drug_classes'] = set(drug_class['drug class name'].str.strip()) - set(generic['drug class'].dropna().str.strip())
unused['indications'] = set(indication['indication name'].str.strip()) - set(generic['indication'].dropna().str.strip())

print("Unused parent entities (exist in reference table but no child references them):")
for tbl, vals in unused.items():
    print(f"  {tbl}: {len(vals)} unused")
    if vals and len(vals) <= 5:
        for v in sorted(vals):
            print(f"    - {v}")
    elif vals:
        for v in sorted(vals)[:3]:
            print(f"    - {v}")
        print(f"    ... and {len(vals)-3} more")

Unused parent entities (exist in reference table but no child references them):
  manufacturers: 8 unused
    - AqVida bangladesh
    - Credence Pharmaceuticals Ltd.
    - D16 Pharma & Biotec
    ... and 5 more
  dosage_forms: 1 unused
    - Bolus Tablet
  generics: 50 unused
    - Artemether
    - Benzocaine + Camphor + Methanol + Phenol
    - Benzyl Alcohol
    ... and 47 more
  drug_classes: 63 unused
    - Abdominal gas adsorbent preparations
    - Adenosine triphosphate-citrate lyase (ACL) inhibitor
    - Anabolic steroid (Androgens)
    ... and 60 more
  indications: 1371 unused
    - AIDS-related Kaposi's sarcoma
    - Abortion
    - Abrasion
    ... and 1368 more


## 8. Duplicate Entity Investigation — Drug Class

A critical data quality issue: the `drug_class` reference table contains **20 names** that appear with **multiple different IDs**. This creates ambiguity in the name-to-ID mapping.

In [11]:
# Identify drug class names with multiple IDs
dc_name_counts = drug_class.groupby('drug class name')['drug class id'].nunique()
dc_duplicates = dc_name_counts[dc_name_counts > 1]

print(f"Drug class names with duplicate IDs: {len(dc_duplicates)}")
print(f"Total duplicate rows: {drug_class[drug_class['drug class name'].isin(dc_duplicates.index)].shape[0]}")
print()

# Show all duplicates with their IDs and generics counts
for name in dc_duplicates.index:
    subset = drug_class[drug_class['drug class name'] == name]
    ids = subset['drug class id'].tolist()
    gen_counts = subset['generics count'].tolist()
    print(f"  '{name}'")
    print(f"    IDs: {ids}")
    print(f"    generics_count: {gen_counts}")
    # Check if generics_count values are all the same (true duplicate)
    if len(set(gen_counts)) == 1:
        print(f"    → TRUE DUPLICATE (identical generics_count across all IDs)")
    else:
        print(f"    → AMBIGUOUS (different generics_count values)")
    print()

# How many generics are affected?
affected_generics = generic[generic['drug class'].isin(dc_duplicates.index)]
print(f"Generics affected by duplicate drug class names: {len(affected_generics)}")
print(f"\nResolution strategy: For SQL, use the LOWEST ID as canonical for each name.")

Drug class names with duplicate IDs: 20
Total duplicate rows: 52

  'Adjunct anti-epileptic drugs'
    IDs: [590, 168]
    generics_count: [13, 11]
    → AMBIGUOUS (different generics_count values)

  'Alpha adrenoceptor blocking drugs'
    IDs: [125, 492]
    generics_count: [5, 5]
    → TRUE DUPLICATE (identical generics_count across all IDs)

  'Alpha and Beta-adrenergic agonist'
    IDs: [493, 126]
    generics_count: [1, 0]
    → AMBIGUOUS (different generics_count values)

  'Angiotensin-converting enzyme (ACE) inhibitors'
    IDs: [495, 118]
    generics_count: [5, 5]
    → TRUE DUPLICATE (identical generics_count across all IDs)

  'Angiotensin-ll receptor blocker'
    IDs: [512, 509, 494, 117]
    generics_count: [7, 7, 7, 7]
    → TRUE DUPLICATE (identical generics_count across all IDs)

  'Antacids'
    IDs: [530, 527, 525, 407]
    generics_count: [6, 6, 8, 6]
    → AMBIGUOUS (different generics_count values)

  'Anti-dyspeptic/Carminatives'
    IDs: [533, 408]
    generics

## 9. Delimiter Investigation

Some drug class and indication values contain commas, ampersands, or slashes. We need to determine whether these are **delimiters** (separating multiple entities) or **part of the name itself**.

In [12]:
# Check drug class values for potential delimiters
dc_vals = generic['drug class'].dropna()
print("=== Drug Class Delimiter Check ===")
for char, label in [(',', 'comma'), ('/', 'slash'), ('&', 'ampersand'), (';', 'semicolon')]:
    count = dc_vals.str.contains(char, regex=False).sum()
    if count > 0:
        print(f"\n  Values containing '{label}' ({char}): {count}")
        examples = dc_vals[dc_vals.str.contains(char, regex=False)].unique()[:3]
        for ex in examples:
            # Check if this exact value exists in the drug_class reference table
            exists = ex.strip() in drug_class['drug class name'].str.strip().values
            print(f"    '{ex}' — exists in drug_class table: {exists}")

print("\n  Conclusion: Commas, slashes, and ampersands are PART OF THE NAME.")
print("  They are NOT delimiters. Each cell contains exactly ONE drug class entity.")
print("  Evidence: every value matches exactly one entry in the drug_class reference table.")

=== Drug Class Delimiter Check ===

  Values containing 'comma' (,): 49
    'Vaccines, Anti-sera & Immunoglobulin' — exists in drug_class table: True
    'Cholagogues, Cholelitholytics & Hepatic Protectors' — exists in drug_class table: True
    'Iron, Vitamin & Mineral Combined preparation' — exists in drug_class table: True

  Values containing 'slash' (/): 40
    'BPH/ Urinary retention/ Urinary incontinence' — exists in drug_class table: True
    'Anticholinergics (antimuscarinics)/ Anti-spasmodics' — exists in drug_class table: True
    'Drugs for HIV / Anti-retroviral drugs' — exists in drug_class table: True

  Values containing 'ampersand' (&): 216
    'Hydrocortisone & Combined preparations' — exists in drug_class table: True
    'Herpes simplex & Varicella-zoster virus infections' — exists in drug_class table: True
    'Vaccines, Anti-sera & Immunoglobulin' — exists in drug_class table: True

  Conclusion: Commas, slashes, and ampersands are PART OF THE NAME.
  They are NOT d

In [13]:
# Check indication values for potential delimiters
ind_vals = generic['indication'].dropna()
print("=== Indication Delimiter Check ===")
for char, label in [(',', 'comma'), (';', 'semicolon'), ('&', 'ampersand')]:
    count = ind_vals.str.contains(char, regex=False).sum()
    if count > 0:
        print(f"\n  Values containing '{label}' ({char}): {count}")
        examples = ind_vals[ind_vals.str.contains(char, regex=False)].unique()[:3]
        for ex in examples:
            exists = ex.strip() in indication['indication name'].str.strip().values
            print(f"    '{ex}' — exists in indication table: {exists}")

print("\n  Conclusion: Same as drug class — special characters are part of the name.")
print("  No multi-value fields detected. No junction tables needed.")

=== Indication Delimiter Check ===

  Values containing 'comma' (,): 11
    'Poisoning due to foods, heavy meals & drugs' — exists in indication table: True
    'Lactose intolerance such as bloating, flatulence, diarrhea' — exists in indication table: True
    'Maintain healthy skin, hair, nail, teeth, bone, eye and nervous system' — exists in indication table: True

  Values containing 'ampersand' (&): 11
    'Poisoning due to foods, heavy meals & drugs' — exists in indication table: True
    'Rough & defective voice' — exists in indication table: True
    'Prevention & control of bleeding in Factor VIII deficiency' — exists in indication table: True

  Conclusion: Same as drug class — special characters are part of the name.
  No multi-value fields detected. No junction tables needed.


## 10. Price & Package Modeling


In [14]:
# Analyze price-related columns
price_cols = ['package_container_raw', 'package_description', 'price', 'pack_price', 'pack_size', 'currency']
print("=== Price/Package Column Summary ===\n")
for col in price_cols:
    na = medicine[col].isna().sum()
    nn = medicine[col].notna().sum()
    pct = na / len(medicine) * 100
    print(f"  {col:25s}  non-null={nn:>6,}  null={na:>5,}  ({pct:.1f}% missing)")

# Is there a 1:1 relationship between brand_id and price?
print(f"\n  brand id is unique: {medicine['brand id'].is_unique}")
print(f"  → Each medicine has at most ONE price row.")
print(f"  → Price is an ATTRIBUTE of medicine, not a separate entity.")

# How many medicines have multiple price entries embedded in the raw string?
taka = chr(2547)  # ৳
multi_price = medicine['package_container_raw'].dropna().str.count(taka)
print(f"\n  Rows with 1 price symbol:  {(multi_price == 1).sum():,}")
print(f"  Rows with 2 price symbols: {(multi_price == 2).sum():,}")
print(f"  Rows with 3+ price symbols: {(multi_price >= 3).sum():,}")
print(f"\n  ⚠️ {(multi_price >= 3).sum()} rows have multiple prices embedded in the raw text.")
print(f"  Only the FIRST price was extracted during cleaning.")
print(f"  Full package/price normalization is deferred to a future phase.")

=== Price/Package Column Summary ===

  package_container_raw      non-null=21,672  null=   42  (0.2% missing)
  package_description        non-null=21,672  null=   42  (0.2% missing)
  price                      non-null=20,357  null=1,357  (6.2% missing)
  pack_price                 non-null=13,527  null=8,187  (37.7% missing)
  pack_size                  non-null=13,527  null=8,187  (37.7% missing)
  currency                   non-null=21,714  null=    0  (0.0% missing)

  brand id is unique: True
  → Each medicine has at most ONE price row.
  → Price is an ATTRIBUTE of medicine, not a separate entity.

  Rows with 1 price symbol:  6,830
  Rows with 2 price symbols: 14,256
  Rows with 3+ price symbols: 550

  ⚠️ 550 rows have multiple prices embedded in the raw text.
  Only the FIRST price was extracted during cleaning.
  Full package/price normalization is deferred to a future phase.


## 11. Name-to-ID Mapping Strategy


In [15]:
# Generate and save mapping tables
# These will be used during SQL data loading to replace name-based FKs with ID-based FKs

# Generic mapping (1:1 — all names are unique)
gen_map = generic[['generic id', 'generic name']].copy()
gen_map.to_csv(MAPPINGS / 'generic_mapping.csv', index=False)
print(f"generic_mapping.csv:      {len(gen_map):>5,} rows  (unique names, clean 1:1)")

# Manufacturer mapping (1:1 — all names are unique)
mfr_map = manufacturer[['manufacturer id', 'manufacturer name']].copy()
mfr_map.to_csv(MAPPINGS / 'manufacturer_mapping.csv', index=False)
print(f"manufacturer_mapping.csv: {len(mfr_map):>5,} rows  (unique names, clean 1:1)")

# Dosage form mapping (1:1 — all names are unique)
dos_map = dosage_form[['dosage form id', 'dosage form name']].copy()
dos_map.to_csv(MAPPINGS / 'dosage_form_mapping.csv', index=False)
print(f"dosage_form_mapping.csv:  {len(dos_map):>5,} rows  (unique names, clean 1:1)")

# Drug class mapping — DEDUPLICATED (use lowest ID per name)
# This resolves the 20 duplicate-name entries by picking the canonical ID
dc_map = drug_class.groupby('drug class name')['drug class id'].min().reset_index()
dc_map.columns = ['drug class name', 'drug class id']
dc_map = dc_map[['drug class id', 'drug class name']].sort_values('drug class id')
dc_map.to_csv(MAPPINGS / 'drug_class_mapping.csv', index=False)
print(f"drug_class_mapping.csv:   {len(dc_map):>5,} rows  (deduplicated — lowest ID per name)")

# Indication mapping (1:1 — all names are unique)
ind_map = indication[['indication id', 'indication name']].copy()
ind_map.to_csv(MAPPINGS / 'indication_mapping.csv', index=False)
print(f"indication_mapping.csv:   {len(ind_map):>5,} rows  (unique names, clean 1:1)")

print(f"\nAll mappings saved to: {MAPPINGS}")

generic_mapping.csv:      1,711 rows  (unique names, clean 1:1)
manufacturer_mapping.csv:   240 rows  (unique names, clean 1:1)
dosage_form_mapping.csv:    113 rows  (unique names, clean 1:1)
drug_class_mapping.csv:     421 rows  (deduplicated — lowest ID per name)
indication_mapping.csv:   2,043 rows  (unique names, clean 1:1)

All mappings saved to: c:\Users\ASUS-PC\Downloads\Telegram Desktop\Project Data Analysis\Project Data Analysis\Project Data Analysis\reports\key_mappings
